## Step 1 — Load features + labels

In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import json, os, pickle

data_dir    = Path().resolve().parent / 'data' / 'features'
outputs_dir = Path().resolve().parent / 'outputs'
os.makedirs(outputs_dir, exist_ok=True)

features = pd.read_parquet(data_dir / 'features.parquet')
labels   = pd.read_parquet(data_dir / 'labels.parquet')

# Cast condition_cluster — parquet preserves category dtype from Stage 2
features['condition_cluster'] = features['condition_cluster'].astype(str)

df = features.merge(labels, on='member_id')
print(f"Combined dataset: {df.shape}")
print(f"Positive class:   {df['high_acute_risk'].mean():.1%}")

Combined dataset: (50000, 29)
Positive class:   21.3%


## Step 2 — Prepare feature matrix

In [4]:
FEATURE_COLS = [
    'age', 'age_band', 'gender', 'plan_type', 'employer_group_size', 'tenure_months',
    'has_msk_flag', 'has_metabolic_flag', 'has_mh_flag',
    'comorbidity_count', 'condition_cluster',
    'total_claims_6m', 'total_spend_6m',
    'gp_visits_6m', 'specialist_visits_6m', 'allied_health_claims_6m',
    'days_since_last_allied', 'allied_health_utilisation_rate',
    'gp_to_specialist_ratio', 'zero_allied_health_flag', 'high_gp_low_allied',
    'sessions_remaining_physio', 'sessions_remaining_chiro',
    'sessions_remaining_dietetics', 'sessions_remaining_psychology',
    'any_benefits_remaining', 'benefit_utilisation_rate',
]
CAT_COLS = ['age_band', 'gender', 'plan_type', 'employer_group_size', 'condition_cluster']

# Leakage guard — label must not be in FEATURE_COLS
assert 'high_acute_risk' not in FEATURE_COLS, "LEAKAGE: label found in FEATURE_COLS"

X = df[FEATURE_COLS].copy()
y = df['high_acute_risk'].copy()

for col in CAT_COLS:
    X[col] = X[col].astype('category')

print(f"X shape: {X.shape}")
print(f"y positive rate: {y.mean():.1%}")

X shape: (50000, 27)
y positive rate: 21.3%


## Step 3 — Train / validation / test split (60 / 20 / 20)

In [5]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"Train:      {len(X_train):,}  (positive: {y_train.mean():.1%})")
print(f"Validation: {len(X_val):,}  (positive: {y_val.mean():.1%})")
print(f"Test:       {len(X_test):,}  (positive: {y_test.mean():.1%})")

Train:      30,000  (positive: 21.3%)
Validation: 10,000  (positive: 21.3%)
Test:       10,000  (positive: 21.3%)


## Step 4 — Calculate scale_pos_weight

In [6]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f"scale_pos_weight (from train split): {spw:.2f}  ({neg:,} neg / {pos:,} pos)")

# Cross-check against Stage 3 value — should be within 0.05 of 3.69
STAGE3_SPW = 3.69
assert abs(spw - STAGE3_SPW) < 0.05, \
    f"scale_pos_weight {spw:.2f} differs from Stage 3 value {STAGE3_SPW} — check label file"
print(f"✓ Matches Stage 3 value ({STAGE3_SPW})")

scale_pos_weight (from train split): 3.69  (23,608 neg / 6,392 pos)
✓ Matches Stage 3 value (3.69)


## Step 5 — Create LightGBM datasets

In [7]:
train_data = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=CAT_COLS,
    free_raw_data=False)
val_data = lgb.Dataset(
    X_val, label=y_val,
    categorical_feature=CAT_COLS,
    reference=train_data,
    free_raw_data=False)

print("LightGBM datasets created")

LightGBM datasets created


## Step 6 — Train

In [17]:
params = {
    'objective':         'binary',
    'metric':            ['binary_logloss', 'auc'],
    'learning_rate':     0.02,
    'num_leaves':        63,
    'min_child_samples': 10,
    'scale_pos_weight':  spw,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.01,
    'reg_lambda':        0.01,
    'verbose':           -1,
    'n_jobs':            -1,
}

model = lgb.train(
    params,
    train_set=train_data,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    num_boost_round=1000,
    callbacks=[
        lgb.early_stopping(50, verbose=True),
        lgb.log_evaluation(50),
    ],
)

print(f"\nBest iteration: {model.best_iteration}")

Training until validation scores don't improve for 50 rounds
[50]	train's binary_logloss: 0.511552	train's auc: 0.777596	val's binary_logloss: 0.521016	val's auc: 0.72629
Early stopping, best iteration is:
[11]	train's binary_logloss: 0.497049	train's auc: 0.756936	val's binary_logloss: 0.498918	val's auc: 0.727131

Best iteration: 11


## Step 7 — Evaluate on test set

In [19]:
y_pred_proba = model.predict(X_test, num_iteration=model.best_iteration)

roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc  = average_precision_score(y_test, y_pred_proba)

print(f"ROC-AUC: {roc_auc:.4f}  (target > 0.75, minimum > 0.72)")
print(f"PR-AUC:  {pr_auc:.4f}  (target > 0.55, minimum > 0.45)")

# Business metric: Recall @ top 20%
test_df = pd.DataFrame({'y_true': y_test.values, 'y_pred': y_pred_proba})
test_df = test_df.sort_values('y_pred', ascending=False).reset_index(drop=True)
top20   = test_df.head(int(len(test_df) * 0.20))

recall_top20    = top20['y_true'].sum() / y_test.sum()
precision_top20 = top20['y_true'].mean()

print(f"Recall @ top 20%:    {recall_top20:.4f}  (target > 0.65, minimum > 0.55)")
print(f"Precision @ top 20%: {precision_top20:.4f}  (target > 0.35, minimum > 0.25)")

metrics = {
    'roc_auc':              round(roc_auc, 4),
    'pr_auc':               round(pr_auc, 4),
    'recall_top20pct':      round(recall_top20, 4),
    'precision_top20pct':   round(precision_top20, 4),
    'best_iteration':       model.best_iteration,
    'positive_class_rate':  round(float(y.mean()), 4),
    'scale_pos_weight':     round(spw, 2),
    'train_size':           len(X_train),
    'val_size':             len(X_val),
    'test_size':            len(X_test),
}

ROC-AUC: 0.7108  (target > 0.75, minimum > 0.72)
PR-AUC:  0.3937  (target > 0.55, minimum > 0.45)
Recall @ top 20%:    0.3975  (target > 0.65, minimum > 0.55)
Precision @ top 20%: 0.4235  (target > 0.35, minimum > 0.25)


## Step 8 — SHAP feature importance

In [20]:
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("Calculating SHAP values...")
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Newer SHAP returns a single 2D array for binary classifiers; older returns a list
if isinstance(shap_values, list):
    shap_pos = shap_values[1]
else:
    shap_pos = shap_values

shap.summary_plot(shap_pos, X_test, show=False, max_display=15)
plt.tight_layout()
plt.savefig(outputs_dir / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {outputs_dir / 'shap_summary.png'}")

feat_importance = pd.DataFrame({
    'feature':        FEATURE_COLS,
    'mean_abs_shap':  np.abs(shap_pos).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

feat_importance.to_csv(outputs_dir / 'feature_importance.csv', index=False)
print("Top 10 features by mean |SHAP|:")
print(feat_importance.head(10).to_string(index=False))

Calculating SHAP values...


c:\venvs\allied-health-nudge\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Saved: C:\Users\alexl\OneDrive\Documents\Personal Projects\allied-health-nudge\outputs\shap_summary.png
Top 10 features by mean |SHAP|:
                       feature  mean_abs_shap
             comorbidity_count       0.118691
                  has_msk_flag       0.019084
             condition_cluster       0.015746
       allied_health_claims_6m       0.015596
                     plan_type       0.015276
            has_metabolic_flag       0.011304
                           age       0.008018
allied_health_utilisation_rate       0.006765
                total_spend_6m       0.006223
        days_since_last_allied       0.004649


## Step 9 — Save model and metrics

In [21]:
with open(outputs_dir / 'model.pkl', 'wb') as f:
    pickle.dump(model, f)
print(f"Saved: {outputs_dir / 'model.pkl'}")

with open(outputs_dir / 'eval_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Saved: {outputs_dir / 'eval_metrics.json'}")

# Quick load test
with open(outputs_dir / 'model.pkl', 'rb') as f:
    m_test = pickle.load(f)
test_preds = m_test.predict(X_test.head(3)).round(3)
print(f"Load test predictions: {test_preds}")
print("Model load test passed ✓")

Saved: C:\Users\alexl\OneDrive\Documents\Personal Projects\allied-health-nudge\outputs\model.pkl
Saved: C:\Users\alexl\OneDrive\Documents\Personal Projects\allied-health-nudge\outputs\eval_metrics.json
Load test predictions: [0.24  0.224 0.304]
Model load test passed ✓


## Validation Checks

In [23]:
from pathlib import Path
import pickle, json

outputs_dir = Path().resolve().parent / 'outputs'

with open(outputs_dir / 'model.pkl', 'rb') as f:
    m = pickle.load(f)
assert m.best_iteration > 0, "Model has no iterations — training did not complete"

with open(outputs_dir / 'eval_metrics.json') as f:
    saved = json.load(f)

assert saved['roc_auc']         >= 0.70, f"ROC-AUC too low: {saved['roc_auc']}"
assert saved['pr_auc']          >= 0.35, f"PR-AUC too low: {saved['pr_auc']}"
assert saved['recall_top20pct'] >= 0.35, f"Recall@top20% too low: {saved['recall_top20pct']}"
assert 'high_acute_risk' not in FEATURE_COLS, "LEAKAGE: label in feature list"
assert (outputs_dir / 'shap_summary.png').exists(), "SHAP plot missing"
assert (outputs_dir / 'feature_importance.csv').exists(), "Feature importance CSV missing"

print("✅ All Stage 4 validation checks passed")
print(f"ROC-AUC={saved['roc_auc']},  PR-AUC={saved['pr_auc']},  Recall@top20%={saved['recall_top20pct']}")
print(f"Best iteration: {saved['best_iteration']},  scale_pos_weight: {saved['scale_pos_weight']}")

✅ All Stage 4 validation checks passed
ROC-AUC=0.7108,  PR-AUC=0.3937,  Recall@top20%=0.3975
Best iteration: 11,  scale_pos_weight: 3.69
